In [ ]:
# Cell 1 — Install
# Do NOT install torch — Colab already has CUDA torch pre-installed
# Installing it here overwrites it with a CPU-only version
!pip install ultralytics -q

from ultralytics import YOLO
import yaml, os, zipfile, random, json, shutil
import torch
from google.colab import drive

# Verify GPU immediately
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU detected — go to Runtime > Change runtime type > T4 GPU, then restart")

In [ ]:
# Cell 2 — Mount Drive, verify GPU, find & extract dataset
import os, zipfile, torch
from google.colab import drive

drive.mount('/content/drive')

# Hard stop if no GPU
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU detected. Go to Runtime > Disconnect and delete runtime > Change runtime type > T4 GPU > Save > Reconnect, then re-run from Cell 1.")

print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
device = 0

# Find the zip anywhere in Drive
ZIP_NAME = 'SVS_Combined.zip'
print(f"\nSearching for {ZIP_NAME} in Drive...")
zip_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if f == ZIP_NAME:
            zip_path = os.path.join(root, f)
            print(f"Found: {zip_path}")
            break
    if zip_path:
        break

if not zip_path:
    # List root of Drive to help locate it
    print("❌ Zip not found. Files in Drive root:")
    for f in os.listdir('/content/drive/MyDrive'):
        print(f"  {f}")
    raise FileNotFoundError(f"Upload {ZIP_NAME} to Google Drive and re-run this cell.")

# Extract
DATASET_ROOT = '/content/dataset'
os.makedirs(DATASET_ROOT, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(DATASET_ROOT)

print(f"\nExtracted to {DATASET_ROOT}")
print(f"Files: {len(os.listdir(DATASET_ROOT))}")

In [ ]:
# Cell 3 — Convert COCO JSON to YOLO segmentation format + train/valid/test split
DATASET_ROOT = '/content/dataset'
JSON_PATH    = f'{DATASET_ROOT}/labels_svs_combined.json'

# Combined 20-class list (COCO id 1..20 -> YOLO index 0..19)
CLASS_NAMES = [
    'data_sticker',       # 0   (COCO id 1)
    'water_filter',       # 1   (COCO id 2)
    'pressure_gauge',     # 2   (COCO id 3)
    'water_line',         # 3   (COCO id 4)
    'water_stop',         # 4   (COCO id 5)
    'qr_code',            # 5   (COCO id 6)
    'aqua_sticker',       # 6   (COCO id 7)
    'installation_date',  # 7   (COCO id 8)
    'drain_grate',        # 8   (COCO id 9)
    'floor_drain',        # 9   (COCO id 10)
    'drain_outlet',       # 10  (COCO id 11)
    'air-gap_fitting',    # 11  (COCO id 12)
    'drain_line_vent',    # 12  (COCO id 13)
    'bin_drain_line',     # 13  (COCO id 14)
    'BackOfMachine',      # 14  (COCO id 15)
    'FrontOfMachine',     # 15  (COCO id 16)
    'IceBin',             # 16  (COCO id 17)
    'IceHead',            # 17  (COCO id 18)
    'IceMaker',           # 18  (COCO id 19)
    'SideOfMachine',      # 19  (COCO id 20)
]

# Load COCO JSON
with open(JSON_PATH) as f:
    coco = json.load(f)

# Build COCO category ID -> YOLO class index mapping
cat_id_to_yolo = {}
for cat in coco['categories']:
    coco_id = cat['id']
    name = cat['name']
    if name in CLASS_NAMES:
        cat_id_to_yolo[coco_id] = CLASS_NAMES.index(name)
    else:
        print(f"⚠️ Unknown category: {name} (id {coco_id})")

print(f"Category mapping (COCO id -> YOLO index):")
for coco_id, yolo_idx in sorted(cat_id_to_yolo.items()):
    print(f"  COCO {coco_id:2d} -> YOLO {yolo_idx:2d}: {CLASS_NAMES[yolo_idx]}")

# Build lookups
img_lookup = {img['id']: img for img in coco['images']}
ann_lookup = {}
for ann in coco['annotations']:
    ann_lookup.setdefault(ann['image_id'], []).append(ann)

# Create output folders
for split in ['train', 'valid', 'test']:
    os.makedirs(f'{DATASET_ROOT}/{split}/images', exist_ok=True)
    os.makedirs(f'{DATASET_ROOT}/{split}/labels', exist_ok=True)

# Shuffle and split (70% train, 20% valid, 10% test)
all_ids = [img['id'] for img in coco['images']]
random.seed(42)
random.shuffle(all_ids)
n = len(all_ids)
train_ids = all_ids[:int(n * 0.70)]
valid_ids  = all_ids[int(n * 0.70):int(n * 0.90)]
test_ids   = all_ids[int(n * 0.90):]

split_map = {}
for iid in train_ids: split_map[iid] = 'train'
for iid in valid_ids:  split_map[iid] = 'valid'
for iid in test_ids:   split_map[iid] = 'test'

skipped = 0
converted = 0
seg_count = 0
bbox_fallback = 0

for img_info in coco['images']:
    iid   = img_info['id']
    fname = img_info['file_name']
    img_w = img_info['width']
    img_h = img_info['height']
    split = split_map[iid]

    src_img = os.path.join(DATASET_ROOT, fname)
    if not os.path.exists(src_img):
        skipped += 1
        continue

    # Copy image
    shutil.copy2(src_img, f'{DATASET_ROOT}/{split}/images/{fname}')

    # Write YOLO label file
    label_name = os.path.splitext(fname)[0] + '.txt'
    with open(f'{DATASET_ROOT}/{split}/labels/{label_name}', 'w') as f:
        for ann in ann_lookup.get(iid, []):
            cat_id = cat_id_to_yolo.get(ann['category_id'])
            if cat_id is None:
                continue  # skip unmapped categories

            # Prefer segmentation polygons if available
            if ann.get('segmentation') and len(ann['segmentation']) > 0 and isinstance(ann['segmentation'][0], list):
                # YOLO seg format: class_id x1 y1 x2 y2 x3 y3 ...
                poly = ann['segmentation'][0]  # first polygon
                normalized = []
                for i in range(0, len(poly), 2):
                    nx = poly[i] / img_w
                    ny = poly[i + 1] / img_h
                    normalized.extend([f'{nx:.6f}', f'{ny:.6f}'])
                f.write(f'{cat_id} {" ".join(normalized)}\n')
                seg_count += 1
            else:
                # Fallback to bbox if no segmentation
                x, y, w, h = ann['bbox']
                x_center = (x + w / 2) / img_w
                y_center = (y + h / 2) / img_h
                f.write(f'{cat_id} {x_center:.6f} {y_center:.6f} {w/img_w:.6f} {h/img_h:.6f}\n')
                bbox_fallback += 1

    converted += 1

print(f'\nConverted: {converted}  |  Skipped: {skipped}')
print(f'Segmentation annotations: {seg_count}  |  Bbox fallbacks: {bbox_fallback}')
for split in ['train', 'valid', 'test']:
    n_img = len(os.listdir(f'{DATASET_ROOT}/{split}/images'))
    n_lbl = len(os.listdir(f'{DATASET_ROOT}/{split}/labels'))
    print(f'  {split}: {n_img} images, {n_lbl} labels')

# Print a sample label to verify
sample = os.listdir(f'{DATASET_ROOT}/train/labels')[0]
print(f'\nSample label ({sample}):')
with open(f'{DATASET_ROOT}/train/labels/{sample}') as f:
    print(f.read())

In [ ]:
# Cell 4 — Create data.yaml
DATASET_ROOT = '/content/dataset'

data = {
    'path': DATASET_ROOT,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 20,
    'names': CLASS_NAMES
}

yaml_path = f'{DATASET_ROOT}/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print('data.yaml created:')
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# Cell 5 — Train (YOLO26s-seg — small model, segmentation, latest architecture)
from ultralytics import YOLO
import os, torch

DATASET_ROOT = '/content/dataset'
PROJECT_DIR  = '/content/drive/MyDrive/yolo_training'
device = 0 if torch.cuda.is_available() else 'cpu'
os.makedirs(PROJECT_DIR, exist_ok=True)

# YOLO26 small segmentation model — latest architecture
model = YOLO('yolo26s-seg.pt')

results = model.train(
    data=f'{DATASET_ROOT}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,            # drop to 8 if out-of-memory
    patience=30,         # early stopping — stops if no improvement for 30 epochs

    # Augmentations:
    mosaic=1.0,
    mixup=0.15,          # blend images — helps with cluttered backgrounds
    copy_paste=0.3,      # seg-specific — copies objects onto new backgrounds
    degrees=15.0,        # rotation
    scale=0.5,           # crop/scale jitter
    hsv_s=0.7,           # saturation (glare/lighting)
    hsv_v=0.5,           # brightness (dark mechanical rooms)
    fliplr=0.5,
    translate=0.1,

    project=PROJECT_DIR,
    name='yolo26s_seg_combined',
    save=True,
    save_period=25,      # checkpoint every 25 epochs
    device=device
)

In [ ]:
# Cell 6 — Validate & Export
from ultralytics import YOLO
import os

DATASET_ROOT = '/content/dataset'
PROJECT_DIR  = '/content/drive/MyDrive/yolo_training'
RUN_NAME     = 'yolo26s_seg_combined'

best = YOLO(f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt')

metrics = best.val(data=f'{DATASET_ROOT}/data.yaml')
print(f'--- Box Metrics ---')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')

# Segmentation metrics
if hasattr(metrics, 'seg'):
    print(f'\n--- Segmentation Metrics ---')
    print(f'Mask mAP50:    {metrics.seg.map50:.4f}')
    print(f'Mask mAP50-95: {metrics.seg.map:.4f}')

# Per-class breakdown
print(f'\n--- Per-Class mAP50 (Box) ---')
class_names = best.names
for i, ap in enumerate(metrics.box.ap50):
    name = class_names.get(i, f'class_{i}')
    flag = ' ⚠️' if ap < 0.5 else ''
    print(f'  {name:20s} {ap:.4f}{flag}')

# Export to ONNX
best.export(format='onnx', imgsz=640)
print(f'\nAll weights and exports saved to: {PROJECT_DIR}/{RUN_NAME}/weights/')

In [ ]:
# Cell 7 — Download model weights locally
from google.colab import files
import shutil, os

PROJECT_DIR = '/content/drive/MyDrive/yolo_training'
RUN_NAME    = 'yolo26s_seg_combined'
WEIGHTS_DIR = f'{PROJECT_DIR}/{RUN_NAME}/weights'

# Download best.pt (the trained model)
print('Downloading best.pt...')
files.download(f'{WEIGHTS_DIR}/best.pt')

# Uncomment to also download ONNX export:
# print('Downloading best.onnx...')
# files.download(f'{WEIGHTS_DIR}/best.onnx')

# Uncomment to download last checkpoint too:
# print('Downloading last.pt...')
# files.download(f'{WEIGHTS_DIR}/last.pt')

print('\n✅ Download complete. Use best.pt for inference in your Laravel app.')

In [ ]:
# Cell 8 — Resume after disconnect (run instead of Cell 5)
from ultralytics import YOLO
import os

PROJECT_DIR  = '/content/drive/MyDrive/yolo_training'
RUN_NAME     = 'yolo26s_seg_combined'

model = YOLO(f'{PROJECT_DIR}/{RUN_NAME}/weights/last.pt')
results = model.train(resume=True)

In [ ]:
# Cell 9 — Quick test on new images (optional)
# Upload test images to /content/test_images/ or point to a folder
from ultralytics import YOLO
from PIL import Image
import glob

PROJECT_DIR = '/content/drive/MyDrive/yolo_training'
RUN_NAME    = 'yolo26s_seg_combined'
model = YOLO(f'{PROJECT_DIR}/{RUN_NAME}/weights/best.pt')

# Run inference on test images
test_images = glob.glob('/content/test_images/*.*')
if not test_images:
    print('No test images found. Upload images to /content/test_images/')
else:
    results = model.predict(test_images, conf=0.25, save=True)
    print(f'Predictions saved. Tested on {len(test_images)} images.')
    for r in results:
        print(f'  {os.path.basename(r.path)}: {len(r.boxes)} detections')